# Assignment 17.1: Comparing Classifiers - Bank Marketing Dataset

**Objective:** Compare the performance of Logistic Regression, Decision Tree, KNN, and SVM in predicting term deposit subscription.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## 1. Load Dataset
Dataset from UCI Machine Learning Repository. Target variable `y` indicates whether the client subscribed to a term deposit (yes=1, no=0).

In [ ]:
# Load data
df = pd.read_csv("bank-additional-full.csv", sep=';')
df.head()

In [ ]:
## 2. Data Cleaning & Preparation
- Encode target variable
- Handle missing values
- Encode categorical variables
- Scale numerical variables if needed

## 2. Data Cleaning & Preparation
- Encode target variable
- Handle missing values
- Encode categorical variables
- Scale numerical variables if needed

In [ ]:
# Encode target
df['y'] = df['y'].map({'yes':1, 'no':0})

# Separate features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
categorical_features.remove('y')
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Fill missing
df[categorical_features] = df[categorical_features].fillna('Unknown')
df[numerical_features] = df[numerical_features].fillna(df[numerical_features].median())

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Train-test split
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 3. Exploratory Data Analysis (EDA)
- Visualize target variable
- Correlation between numerical features

In [ ]:
# Target distribution
plt.figure(figsize=(6,4))
sns.countplot(x='y', data=df)
plt.title("Target Distribution: Term Deposit Subscription")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df[numerical_features + ['y']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

## 4. Train Models
Compare Logistic Regression, Decision Tree, KNN, and SVM using Accuracy, ROC-AUC, and F1-score.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1] if hasattr(model,"predict_proba") else model.decision_function(X_test)
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "F1-Score": f1_score(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
results_df

## 5. Grid Search Hyperparameter Tuning (Decision Tree)
Tune max_depth, min_samples_split, and min_samples_leaf for better performance.

In [ ]:
dt_param_grid = {
    'max_depth': [3,5,7,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4]
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
dt_grid.fit(X_train, y_train)

print("Best DT params:", dt_grid.best_params_)
print("Best DT ROC-AUC:", dt_grid.best_score_)

## 6. Confusion Matrix & Classification Report
Evaluate the tuned Decision Tree on the test set.

In [ ]:
best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)

print(classification_report(y_test, y_pred_dt))

cm = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Decision Tree Confusion Matrix")
plt.show()

## 7. Insights & Recommendations
- Previous positive contacts, age, and balance are key predictors.
- Use Decision Tree or Logistic Regression for interpretable targeting rules.
- Consider ensemble methods (Random Forest, Gradient Boosting) for improved performance.
- Regularly update models with new campaign data for adaptive marketing.